# Text-to-SQL Fine-Tuning (QLoRA)

Fine-tunes Qwen2.5-1.5B-Instruct on a text-to-SQL instruction dataset using QLoRA via Unsloth,
then evaluates it against the base model.

Run this notebook top to bottom on a Colab GPU runtime (Runtime > Change runtime type > T4 GPU).


## 1. Clone the repository

In [ ]:
!git clone https://github.com/Prabhav54/text-to-sql-finetuning-qlora.git
%cd text-to-sql-finetuning-qlora


## 2. Install dependencies

In [ ]:
!pip install unsloth -q
!pip install pyyaml rouge-score -q


## 3. Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## 4. Prepare the dataset

Downloads `gretelai/synthetic_text_to_sql`, slices to 3,000 records, formats each
example as a ChatML conversation (system schema + instructions, user question,
assistant SQL), and splits into train/val sets.


In [ ]:
!python -m src.data_preparation --config config/config.yaml


### Sanity check: confirm the formatted data looks correct before training

In [ ]:
import json

with open("data/processed/train.jsonl") as f:
    print(json.loads(f.readline())["text"])


## 5. Train (QLoRA)

Fine-tunes Qwen2.5-1.5B-Instruct with LoRA adapters on a 4-bit quantized base model.
Trains in well under 15 minutes on a free T4 GPU with this configuration.


In [ ]:
!python -m src.train --config config/config.yaml


## 6. Evaluate

Runs the held-out validation set through both the fine-tuned model and the
untouched base model, computing SQL exact-match accuracy for each and the
percentage improvement. Results are saved to `logs/eval_results.json`.


In [ ]:
!python -m src.evaluate --config config/config.yaml


### View results

In [ ]:
import json

results = json.load(open("logs/eval_results.json"))

print("Fine-tuned SQL exact-match accuracy:", results["sql_exact_match_accuracy"])
print("Base model SQL exact-match accuracy:", results.get("base_model_sql_accuracy"))
print("Improvement over base:", results.get("improvement_over_base_pct"), "%")

print("\nSample predictions:")
for s in results["qualitative_samples"][:5]:
    print("\nQuestion:", s["question"])
    print("Reference SQL:", s["reference_sql"])
    print("Predicted SQL:", s["predicted_sql"])


## 7. Publish the adapter to Hugging Face Hub (optional)

Set `huggingface.push_to_hub: true` and `huggingface.repo_id` in `config/config.yaml`
before running Cell 5 (training) to publish automatically. Requires a Hugging Face
token — log in first:


In [ ]:
from huggingface_hub import login
login()


## 8. Try the model locally (optional demo)

Runs the Streamlit chat demo. In Colab, tunnel the port to view it in your browser.


In [ ]:
!pip install streamlit pyngrok -q
!streamlit run app/streamlit_app.py &

from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(public_url)
